In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

SupportIQ Day 2


In [2]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

In [3]:
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

display(train_df.head())

Training shape: (10003, 2)
Testing shape: (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
print(train_df.columns)
print(test_df.columns)

Index(['text', 'category'], dtype='str')
Index(['text', 'category'], dtype='str')


In [5]:
print(train_df.isnull().sum())
print(test_df.isnull().sum())

text        0
category    0
dtype: int64
text        0
category    0
dtype: int64


In [ ]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
sample_text = "PLEASE help!!! My card isn't working "

print("Original:")
print(sample_text)

print("\nCleaned:")
print(clean_text(sample_text))

Original:
PLEASE help!!! My card isn't working 

Cleaned:
please help my card isn t working


In [8]:
sample_text = """
I can't withdraw cash from my account.
Please visit https://example.com
Contact me at customer@example.com
"""

print(clean_text(sample_text))

i can t withdraw cash from my account please visit contact me at


In [9]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [10]:
display(
    train_df[["text", "clean_text", "category"]].head(10)
)

,text,clean_text,category
0,I am still waiting on my card?,i am still waiting on my card,card_arrival
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasn t arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,i have been waiting over a week is the card st...,card_arrival
3,Can I track my card while it is in the process...,can i track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",how do i know if i will get my card or if it i...,card_arrival
5,When did you send me my new card?,when did you send me my new card,card_arrival
6,Do you have info about the card on delivery?,do you have info about the card on delivery,card_arrival
7,What do I do if I still have not received my n...,what do i do if i still have not received my n...,card_arrival
8,Does the package with my card have tracking?,does the package with my card have tracking,card_arrival
9,I ordered my card but it still isn't here,i ordered my card but it still isn t here,card_arrival


In [12]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [13]:
X_train = tfidf.fit_transform(train_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])

In [14]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (10003, 10292)
X_test shape: (3080, 10292)


In [15]:
print("Vocabulary size:", len(tfidf.vocabulary_))

Vocabulary size: 10292


In [16]:
print("Example features:")
print(tfidf.get_feature_names_out()[:50])

Example features:
['00' '00 in' '000' '10' '10 minutes' '10 pounds' '100' '100 all'
 '100 but' '18' '18 to' '18 years' '1l' '20' '20 00' '20 but' '20 came'
 '20 what' '20 when' '200' '2018' '30' '30 pounds' '30 what' '3d'
 '3d secure' '40' '500' '500 cash' '500 that' '5x' '80' 'able' 'able to'
 'about' 'about an' 'about atm' 'about changing' 'about days'
 'about direct' 'about disposable' 'about doing' 'about exchange'
 'about getting' 'about half' 'about how' 'about it' 'about me' 'about my'
 'about ordering']


In [17]:
sample_vector = X_train[0]

print(sample_vector)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (1, 10292)>
  Coords	Values
  (0, 411)	0.25045820055601165
  (0, 7633)	0.2666186934917499
  (0, 9346)	0.3563164334513188
  (0, 5939)	0.2126467911172315
  (0, 5387)	0.10310218897065936
  (0, 1633)	0.1439997029450656
  (0, 449)	0.4227165496994996
  (0, 7661)	0.4086243549732222
  (0, 9348)	0.4673966018703363
  (0, 5964)	0.2532247916519013
  (0, 5406)	0.1854551274622703


In [18]:
print("Number of non-zero features:", sample_vector.nnz)

Number of non-zero features: 11


In [19]:
train_processed = train_df[["text", "clean_text", "category"]]
test_processed = test_df[["text", "clean_text", "category"]]

In [20]:
train_processed.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

test_processed.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

In [23]:
y_train = train_df["category"]
y_test = test_df["category"]